# Patient Allocation - Probabilistic Two-Step Workflow

**Purpose**: Allocate population to HSAs using a probabilistic gravity model

## Two-Step Allocation Process

### Step 1: Pixel → ALL Facilities (Probabilistic)
Instead of assigning each population pixel to ONE facility (hard assignment), this workflow
SPLITS each pixel's population across ALL reachable facilities based on gravity model probabilities.

**Gravity Model**:
$$
P(facility_i | pixel) = \frac{Volume_i^\alpha / Distance_i^\beta}{\sum_j Volume_j^\alpha / Distance_j^\beta}
$$

Where:
- $\alpha = 0.75$ (facility size weight)
- $\beta = 1.5$ (distance decay)
- Maximum distance = 100 km

### Step 2: Facility → HSA Aggregation (Three-Case Logic)

After allocating population to all 188 facilities, we aggregate to HSAs using:

| Case | Condition | Assignment |
|------|-----------|------------|
| **Case 1** | Facility inside exactly 1 HSA | 100% to that HSA |
| **Case 2** | Facility outside all HSAs | Assign to nearest HSA anchor within 100km |
| **Case 3** | Facility inside 2+ overlapping HSAs | Proportional using gravity model ($Volume^\alpha / Distance^\beta$) |

Facilities beyond 100km from any HSA anchor are excluded and reported.

---

In [ ]:
# ── BOUNDARY VERSION SELECTOR ──────────────────────────────────────────────
# Choose which HSA boundary bundle to use for this analysis.
# Must match a bundle produced by HSA_FINAL.ipynb.
#   'v6'  — original greedy algorithm (no post-selection corrections)
#   'v7'  — + anchor upgrade/demotion + major-orphan promotion
#   'v8'  — + satellite bubble boundaries
BOUNDARY_VERSION = "v7"   # change as needed
# ────────────────────────────────────────────────────────────────────────────


In [ ]:
%load_ext autoreload
%autoreload 2

# ============================================================================
# CONFIGURATION
# ============================================================================

from pathlib import Path
import os
import pandas as pd
import geopandas as gpd
import numpy as np
from patient_allocation import PatientAllocator, allocate_patients_probabilistic

# Directories
PIPELINE_VERSION = os.environ.get("PIPELINE_VERSION", "v7")
DATA_DIR = Path(os.environ.get("HSA_DATA_DIR", "data"))
OUT_DIR = Path(os.environ.get("HSA_OUT_DIR", os.environ.get("PIPELINE_OUT_DIR", "out")))
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================================
# SELECT NETWORK AND HSA MODE
# ============================================================================
NETWORK = "INF"  # Options: 'INF' or 'NCD'

# Available HSA modes from HSA_v7_FINAL.ipynb:
#   - 'fewest': Minimum number of HSAs
#   - 'footprint': Maximum geographic diversity
#   - 'distance': Minimize travel distance
#   - 'governorate_tau_coverage': 90% coverage per governorate
#   - 'governorate_fewest': One anchor per governorate + minimize total

HSA_MODE = "footprint"  # <- CHANGE THIS to select your HSA mode

# Population raster
POP_RASTER = DATA_DIR / "jor_ppp_2020_UNadj.tif"

# ============================================================================
# GRAVITY MODEL PARAMETERS
# ============================================================================
GRAVITY_PARAMS = {
    'alpha': 0.75,          # Facility size weight
    'beta': 1.5,            # Distance decay exponent
    'max_distance_km': 100, # Absolute ceiling; Case 2 fallback guard is stricter
    'sample_rate': 1        # Set to 1 for production, >1 for testing
}

FALLBACK_GUARD_PARAMS = {
    'fallback_radius_multiplier': 1.5,
    'fallback_min_distance_km': 30.0,
    'major_facility_pop_threshold': 25000.0,
    'major_facility_volume_threshold': None,
    'major_facility_volume_quantile': 0.80,
    'require_same_governorate_for_major': True,
}

print("="*80)
print("PROBABILISTIC PATIENT ALLOCATION CONFIGURATION")
print("="*80)
print(f"Network: {NETWORK}")
print(f"HSA Mode: {HSA_MODE}")
print(f"Population Raster: {POP_RASTER.name}")
print(f"\nGravity Model Parameters:")
for key, val in GRAVITY_PARAMS.items():
    print(f"  {key}: {val}")
print(f"\nFallback Guard Parameters:")
for key, val in FALLBACK_GUARD_PARAMS.items():
    print(f"  {key}: {val}")
print("\nAllocation Mode: PROBABILISTIC (two-step)")
print("="*80)

## 1. Load Data

In [ ]:
# Load HSA boundaries (output from HSA_v7_FINAL.ipynb)
hsa_file = OUT_DIR / f"{NETWORK}_{HSA_MODE}_hsas_{BOUNDARY_VERSION}.geojson"

if not hsa_file.exists():
    raise FileNotFoundError(
        f"HSA file not found: {hsa_file}\n"
        f"Please run HSA_v7_FINAL.ipynb first to generate HSA boundaries for mode '{HSA_MODE}'."
    )

hsas = gpd.read_file(hsa_file)
print(f"Loaded {len(hsas)} HSA anchors from {hsa_file.name}")

# Display HSA anchors
print("\nHSA Anchor Facilities:")
for idx, row in hsas.iterrows():
    fac_name = row.get('HealthFacility', row.get('FacilityName', f'Facility_{idx}'))
    radius = row.get('service_radius_km', row.get('initial_radius_km', 'N/A'))
    print(f"  {idx+1}. {fac_name} (radius: {radius:.1f} km)")

In [ ]:
# Load ALL facilities (not just HSA anchors)
facilities_csv = OUT_DIR / f"{NETWORK}_Facilities_Climate_Features_with_clusters.csv"

if not facilities_csv.exists():
    raise FileNotFoundError(
        f"Facilities file not found: {facilities_csv}\n"
        f"Please ensure the climate features CSV is in the output directory."
    )

all_facilities_df = pd.read_csv(facilities_csv)

# Normalize whitespace in facility names
if 'FacilityName' in all_facilities_df.columns:
    all_facilities_df['HealthFacility'] = all_facilities_df['FacilityName'].str.replace(r'\s+', ' ', regex=True).str.strip()
elif 'HealthFacility' in all_facilities_df.columns:
    all_facilities_df['HealthFacility'] = all_facilities_df['HealthFacility'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Load diagnosis counts for patient volumes
diagnosis_csv = OUT_DIR / f"{NETWORK}_{HSA_MODE}_diagnosis_counts_pivot.csv"

if diagnosis_csv.exists():
    diagnosis_df = pd.read_csv(diagnosis_csv)
    diagnosis_df['healthfacility'] = diagnosis_df['healthfacility'].str.replace(r'\s+', ' ', regex=True).str.strip()
    
    merge_cols = ['healthfacility', 'total_diagnoses']
    for optional_col in ['healthfacilitytype', 'governorate']:
        if optional_col in diagnosis_df.columns:
            merge_cols.append(optional_col)

    all_facilities_df = all_facilities_df.merge(
        diagnosis_df[merge_cols],
        left_on='HealthFacility',
        right_on='healthfacility',
        how='left'
    )
    all_facilities_df['Total'] = all_facilities_df['total_diagnoses'].fillna(1000)  # Default for missing
else:
    print(f"WARNING: Diagnosis counts not found at {diagnosis_csv}")
    print("  Using uniform facility weights (Total = 1000 for all facilities)")
    all_facilities_df['Total'] = 1000

# Create GeoDataFrame
all_facilities = gpd.GeoDataFrame(
    all_facilities_df,
    geometry=gpd.points_from_xy(all_facilities_df['lon'], all_facilities_df['lat']),
    crs='EPSG:4326'
)

print(f"\nLoaded {len(all_facilities)} total facilities")
print(f"  Facilities with volume > 0: {(all_facilities['Total'] > 0).sum()}")
print(f"  Total patient volume: {all_facilities['Total'].sum():,.0f}")

## 2. Step 1: Probabilistic Allocation to ALL Facilities

Each population pixel is allocated to ALL reachable facilities based on gravity model probabilities.

In [ ]:
# Initialize allocator with ALL facilities
print("Initializing PatientAllocator with ALL facilities...")
allocator = PatientAllocator(
    pop_raster_path=str(POP_RASTER),
    facilities_gdf=all_facilities,
    params=GRAVITY_PARAMS
)

print(f"\n  Allocating to {len(all_facilities)} facilities (not just {len(hsas)} HSA anchors)")

In [ ]:
# Run probabilistic allocation (parallel for speed)
print("\nRunning probabilistic allocation...")
print("  (This allocates each pixel's population to ALL reachable facilities)")

facility_allocations, pixel_allocations = allocator.allocate_all_pixels_probabilistic_parallel()

print(f"\nFacility allocation summary:")
print(f"  Facilities with allocated population: {len(facility_allocations)}")
print(f"  Total allocated population: {facility_allocations['allocated_population'].sum():,.0f}")

# Show top 15 facilities
print(f"\nTop 15 facilities by allocated population:")
print(facility_allocations.head(15).to_string(index=False))

print(f"\nPixel allocations tracked: {len(pixel_allocations):,}")

## 3. Step 2: Aggregate Facilities to HSAs

Now aggregate the facility-level populations to HSAs using three-case logic:
- **Case 1**: Facility inside exactly 1 HSA → 100% to that HSA
- **Case 2**: Facility outside all HSAs → nearest HSA anchor within 100km
- **Case 3**: Facility inside 2+ overlapping HSAs → proportional using gravity model (Volume^α / Distance^β)

In [ ]:
# Aggregate facilities to HSAs using three-case logic (with gravity model for Case 3)
hsa_summary, facility_assignments = allocator.aggregate_facilities_to_hsas(
    facility_allocations,
    hsas,
    all_facilities,
    network_type=NETWORK,
    optimization_mode=HSA_MODE,
    max_assignment_distance_km=GRAVITY_PARAMS['max_distance_km'],
    fallback_radius_multiplier=FALLBACK_GUARD_PARAMS['fallback_radius_multiplier'],
    fallback_min_distance_km=FALLBACK_GUARD_PARAMS['fallback_min_distance_km'],
    major_facility_pop_threshold=FALLBACK_GUARD_PARAMS['major_facility_pop_threshold'],
    major_facility_volume_threshold=FALLBACK_GUARD_PARAMS['major_facility_volume_threshold'],
    major_facility_volume_quantile=FALLBACK_GUARD_PARAMS['major_facility_volume_quantile'],
    require_same_governorate_for_major=FALLBACK_GUARD_PARAMS['require_same_governorate_for_major'],
    alpha=GRAVITY_PARAMS['alpha'],
    beta=GRAVITY_PARAMS['beta']
)

In [ ]:
# Display facility assignment breakdown by case
print("\nFacility Assignment Breakdown by Case:")
case_summary = facility_assignments.groupby('assignment_case').agg(
    count=('facility_id', 'count'),
    total_population=('allocated_population', 'sum')
).reset_index()
case_summary['pct_facilities'] = (case_summary['count'] / case_summary['count'].sum() * 100).round(1)
case_summary['pct_population'] = (case_summary['total_population'] / case_summary['total_population'].sum() * 100).round(1)
print(case_summary.to_string(index=False))

## 4. Save Results

In [ ]:
# Save HSA population summary (PRIMARY OUTPUT)
hsa_output = OUT_DIR / f"{NETWORK}_{HSA_MODE}_hsa_populations_probabilistic.csv"
hsa_summary.to_csv(hsa_output, index=False)
print(f"Saved HSA population summary: {hsa_output}")

# Save facility allocations from pixels
fac_alloc_output = OUT_DIR / f"{NETWORK}_{HSA_MODE}_facility_allocations_probabilistic.csv"
facility_allocations.to_csv(fac_alloc_output, index=False)
print(f"Saved facility allocations: {fac_alloc_output}")

# Save facility-to-HSA assignments (shows three-case logic)
fac_assign_output = OUT_DIR / f"{NETWORK}_{HSA_MODE}_facility_hsa_assignments_{BOUNDARY_VERSION}.csv"
facility_assignments.to_csv(fac_assign_output, index=False)
print(f"Saved facility-to-HSA assignments: {fac_assign_output}")

# Save pixel-level allocations (two formats for compatibility)
pixel_output = OUT_DIR / f"pixel_allocations_{NETWORK}_{HSA_MODE}.csv"
pixel_allocations.to_csv(pixel_output, index=False)
print(f"Saved pixel allocations: {pixel_output}")

# Also save as allocation_details.csv (expected by downstream scripts)
details_output = OUT_DIR / f"{NETWORK}_{HSA_MODE}_allocation_details.csv"
pixel_allocations.to_csv(details_output, index=False)
print(f"Saved allocation details: {details_output}")
print(f"  File size: {details_output.stat().st_size / (1024*1024):.1f} MB")

## 5. Paper Tables

In [ ]:
# TABLE 1: HSA Population Summary
print("\n" + "="*80)
print("TABLE 1: HSA Population Summary (Probabilistic Allocation)")
print("="*80)
print(f"Network: {NETWORK} | HSA Mode: {HSA_MODE}")
print("-"*80)

table1 = hsa_summary[['anchor_id', 'anchor_name', 'allocated_patients', 'num_facilities_in_hsa']].copy()
table1['allocated_patients'] = table1['allocated_patients'].apply(lambda x: f"{x:,.0f}")
table1.columns = ['Rank', 'HSA Anchor', 'Allocated Population', 'Facilities']
print(table1.to_string(index=False))

print("-"*80)
total_pop = hsa_summary['allocated_patients'].sum()
total_fac = hsa_summary['num_facilities_in_hsa'].sum()
print(f"{'TOTAL':<6} {'':<45} {total_pop:>20,.0f} {total_fac:>12}")
print("="*80)

In [ ]:
# TABLE 2: Facility Assignment Case Distribution
print("\n" + "="*80)
print("TABLE 2: Facility Assignment Case Distribution")
print("="*80)

case_table = facility_assignments.groupby('assignment_case').agg(
    facilities=('facility_id', 'count'),
    population=('allocated_population', 'sum')
).reset_index()

total_fac = case_table['facilities'].sum()
total_pop = case_table['population'].sum()

case_table['pct_facilities'] = (case_table['facilities'] / total_fac * 100).round(1)
case_table['pct_population'] = (case_table['population'] / total_pop * 100).round(1)
case_table['population'] = case_table['population'].apply(lambda x: f"{x:,.0f}")

print(case_table.to_string(index=False))
print("="*80)

In [ ]:
# TABLE 3: Top 20 Facilities by Allocated Population
print("\n" + "="*80)
print("TABLE 3: Top 20 Facilities by Allocated Population")
print("="*80)

top20 = facility_assignments.nlargest(20, 'allocated_population')[[
    'facility_id', 'allocated_population', 'assignment_case', 'primary_hsa'
]].copy()
top20['allocated_population'] = top20['allocated_population'].apply(lambda x: f"{x:,.0f}")
top20.columns = ['Facility', 'Population', 'Assignment Case', 'Primary HSA']
print(top20.to_string(index=False))
print("="*80)

In [ ]:
# Validation: Compare with raster total
import rasterio

with rasterio.open(POP_RASTER) as src:
    pop_data = src.read(1, masked=True)
    total_pop_raster = float(pop_data.sum())

total_hsa_pop = hsa_summary['allocated_patients'].sum()
excluded_pop = facility_assignments[facility_assignments['excluded']]['allocated_population'].sum()
total_facility_pop = facility_allocations['allocated_population'].sum()

print("\n" + "="*80)
print("VALIDATION: Population Coverage")
print("="*80)
print(f"Total population in raster:     {total_pop_raster:>15,.0f}")
print(f"Total allocated to facilities:  {total_facility_pop:>15,.0f} ({total_facility_pop/total_pop_raster*100:.1f}%)")
print(f"Total assigned to HSAs:         {total_hsa_pop:>15,.0f} ({total_hsa_pop/total_pop_raster*100:.1f}%)")
print(f"Excluded (beyond max dist):     {excluded_pop:>15,.0f} ({excluded_pop/total_pop_raster*100:.2f}%)")
print("="*80)

In [ ]:
print("\n" + "="*80)
print("PROBABILISTIC ALLOCATION COMPLETE")
print("="*80)
print(f"\nOutput files saved to {OUT_DIR}:")
print(f"  - {hsa_output.name}")
print(f"  - {fac_alloc_output.name}")
print(f"  - {fac_assign_output.name}")
print(f"  - {pixel_output.name} (for downstream compatibility)")

In [ ]:
# AUTO_NOTEBOOK_SUMMARY_V1
from pathlib import Path
from datetime import datetime
import os
import json

NOTEBOOK_NAME = "Patient_Allocation_Probabilistic_v2"
NETWORK = globals().get('NETWORK', os.environ.get('NETWORK', 'INF'))
HSA_MODE = globals().get('HSA_MODE', os.environ.get('HSA_MODE', ''))

suffix = f"{NETWORK}_{HSA_MODE}" if HSA_MODE else f"{NETWORK}"

out_root = Path(globals().get('OUT_DIR', globals().get('OUT_ROOT', os.environ.get('HSA_OUT_DIR', os.environ.get('PIPELINE_OUT_DIR', f"out_{os.environ.get('PIPELINE_VERSION', 'v7')}")))))
summary_dir = out_root / 'textresults'
summary_dir.mkdir(parents=True, exist_ok=True)
summary_path = summary_dir / f"{NOTEBOOK_NAME}_{suffix}_results.md"

files = [p for p in out_root.rglob('*') if p.is_file() and p.suffix.lower() in {'.csv', '.json', '.md', '.txt', '.png', '.pdf', '.geojson', '.parquet'}]
files.sort(key=lambda p: p.stat().st_mtime, reverse=True)
important = files[:120]

NL = "\n"

blocks = []
blocks.append(f"# Notebook Results: {NOTEBOOK_NAME}")

meta = [
    f"- Generated: {datetime.now().isoformat(timespec='seconds')}",
    f"- Network: {NETWORK}",
]
if HSA_MODE:
    meta.append(f"- HSA mode: {HSA_MODE}")
blocks.append(NL.join(meta))

file_lines = ['## Important Output Files', '']
for p in important:
    file_lines.append(f"- `{p}`")
blocks.append(NL.join(file_lines))

nb_path = Path(f"{NOTEBOOK_NAME}.ipynb")
if nb_path.exists():
    try:
        nb_data = json.loads(nb_path.read_text())
        blocks.append('## Displayed Cell Outputs')

        for idx, cell in enumerate(nb_data.get('cells', []), start=1):
            if cell.get('cell_type') != 'code':
                continue
            outputs = cell.get('outputs', []) or []
            if not outputs:
                continue

            blocks.append(f"### Cell {idx}")

            for out in outputs:
                otype = out.get('output_type')

                if otype == 'stream':
                    text = ''.join(out.get('text', [])) if isinstance(out.get('text', []), list) else str(out.get('text', ''))
                    text = text.rstrip()
                    if text:
                        blocks.append("```text" + NL + text + NL + "```")

                elif otype in ('execute_result', 'display_data'):
                    data = out.get('data', {})
                    if 'text/markdown' in data:
                        md = ''.join(data['text/markdown']) if isinstance(data['text/markdown'], list) else str(data['text/markdown'])
                        md = md.rstrip()
                        if md:
                            blocks.append(md)
                    elif 'text/plain' in data:
                        txt = ''.join(data['text/plain']) if isinstance(data['text/plain'], list) else str(data['text/plain'])
                        txt = txt.rstrip()
                        if txt:
                            blocks.append("```text" + NL + txt + NL + "```")
                    elif 'text/html' in data:
                        html = ''.join(data['text/html']) if isinstance(data['text/html'], list) else str(data['text/html'])
                        html = html.rstrip()
                        if html:
                            blocks.append("```html" + NL + html + NL + "```")
                    elif 'image/png' in data or 'image/jpeg' in data:
                        blocks.append('_[Image output omitted in text summary]_')

                elif otype == 'error':
                    tb = out.get('traceback', []) or []
                    if tb:
                        err = NL.join(str(t) for t in tb)
                    else:
                        err = f"{out.get('ename', 'Error')}: {out.get('evalue', '')}"
                    blocks.append("```text" + NL + err + NL + "```")

    except Exception as e:
        blocks.append('## Displayed Cell Outputs')
        blocks.append(f"Could not parse notebook outputs: {e}")

summary = (NL + NL).join(b for b in blocks if b and str(b).strip()) + NL
summary_path.write_text(summary)
print(f"Saved notebook summary: {summary_path}")
